# UABR — Phase 4: teach `<ANSWER>` vs `<CALL>` (delegation) on top of `base_v1`

**What happens:**
1. **Knowledge labels.** Ask `base_v1` every non-test fact as a cloze. If it answers correctly, label ANSWER; otherwise CALL. Phase 3 showed that exposure alone does not mean the model knows the fact.
2. **Delegation training.** About 3 epochs on delegation_train with a 50/50 ANSWER/CALL mix. Loss covers only the assistant tokens; the evidence text is masked.
3. **Gate on dev:** both actions learned, routing AUROC clearly above 0.5, the direct path works on known facts, and the CALL path works.

`QUICK = True` → Run All (~5 min) checks the pipeline.

In [ ]:
QUICK = True
RUN_SECOND_SEED = False   # full run only: repeat training with seed 2027 to check reproducibility (+~15 min)
REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"

import os, glob, json, subprocess
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/uabr_slm"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

data_hits = glob.glob("/kaggle/input/**/data_v1/manifest.json", recursive=True)
base_hits = glob.glob("/kaggle/input/**/base_v1.pt", recursive=True)
assert data_hits, "data_v1 not found: Add Input -> Your Work -> the Phase 2 notebook"
assert base_hits, "base_v1.pt not found: Add Input -> Your Work -> the Phase 3 notebook"
DATA, BASE = os.path.dirname(data_hits[0]), base_hits[0]
KNOW = f"{WORK}/facts_knowledge.parquet"
OUT = f"{WORK}/runs/delegation_quick" if QUICK else f"{WORK}/runs/delegation"
print("DATA :", DATA); print("BASE :", BASE, f"({os.path.getsize(BASE) / 1e6:.0f} MB)"); print("OUT  :", OUT)
!git log --oneline -1
!python -m pytest tests -q -x

## 1. Knowledge labels from `base_v1` (≈2 min)
Expected pattern, as in the Phase 3 probe: about 75% known at 80 statements, about 16% at 20, about 4% at 5 and when withheld.

In [ ]:
!python scripts/label_knowledge.py --base {BASE} --data {DATA} --out {KNOW}

## 2. Delegation training + dev gate
Prints dev metrics before training and after every epoch, then the full gate.

In [ ]:
quick = "--quick" if QUICK else ""
!rm -rf {OUT}
!python scripts/train_delegation.py --base {BASE} --data {DATA} --knowledge {KNOW} --out {OUT} {quick}

## 3. Look at real predictions

In [ ]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 40)
df = pd.read_parquet(f"{OUT}/dev_predictions.parquet")
cols = ["question", "gold", "predicted_action", "call_score", "answer_text", "reward_answer", "call_answer_text", "reward_call"]
print("Known-looking facts the model chose to ANSWER:")
print(df[df.predicted_action == "ANSWER"].sort_values("call_score").head(8)[cols].to_string(index=False))
print("\nFacts the model chose to CALL for:")
print(df[df.predicted_action == "CALL"].sort_values("call_score", ascending=False).head(8)[cols].to_string(index=False))
print("\nBy exposure tier (share routed to CALL, direct accuracy, CALL accuracy):")
print(df.groupby("exposure_count").agg(call_rate=("predicted_action", lambda s: (s == "CALL").mean()),
      direct_acc=("reward_answer", "mean"), call_acc=("reward_call", "mean"), n=("prompt_id", "size")).round(3).to_string())

## 4. Optional: second seed (reproducibility gate)

In [ ]:
if RUN_SECOND_SEED and not QUICK:
    !python scripts/train_delegation.py --base {BASE} --data {DATA} --knowledge {KNOW} --out {WORK}/runs/delegation_seed2027 --seed 2027
    a = json.load(open(f"{OUT}/gate_report.json")); b = json.load(open(f"{WORK}/runs/delegation_seed2027/gate_report.json"))
    print(f"AUROC seed 2026 {a['auroc']:.3f} vs seed 2027 {b['auroc']:.3f}")
else:
    print("skipped")